# 12 · GSE232381 · bulk_RNA_seq · normalise

Reads the authors' Excel file, `authors_columns.rds` and `metadata.rds` (notebook 11), and NCBI's gene table (gene_info).
Writes `data/run_artifacts/GSE232381/expression.rds`.

1. **The authors' expected counts** for the 25 samples, one row per gene symbol.
2. **Current NCBI symbols**, the same names as GSE65391 (NCBI Gene: Brown GR et al. Gene: a gene-centered
   information resource at NCBI. *Nucleic Acids Res* 2015;43:D36-D42).
3. **Filter low counts** (WGCNA FAQ): remove genes with a count below 10 in more than 90% of samples.
4. **PFlog1pPF** (Booeshaghi AS, Hallgrímsdóttir IB, Gálvez-Merchán Á, Pachter L. "Depth
   normalization for single-cell genomics count data", bioRxiv 2022, doi:10.1101/2022.05.06.490859):
   - PF: scale each sample's counts so its total equals the mean total over samples;
   - log1p: log(1 + x);
   - PF again on the log values.

## Read the authors' expected counts

In [ ]:
source("/home/jovyan/session_data/Attempt0/src/paths.R")
suppressMessages(library(readxl))
meta <- readRDS(art("GSE232381", "metadata.rds"))
cols <- readRDS(art("GSE232381", "authors_columns.rds"))
x <- read_excel(raw("GSE232381", "GSE232381_Processed_data_files-Active_LN_vs_Inactive_LN.xlsx"), sheet = "Detail",
                .name_repair = "minimal")
ec <- cols[cols$measure == "Expected count", ]
counts <- as.matrix(x[, ec$position]); dimnames(counts) <- list(x$GeneID, ec$gsm)
counts <- counts[, meta$sample]
rm(x)
stopifnot(!anyNA(counts), all(counts >= 0), !anyDuplicated(rownames(counts)))
c(genes = nrow(counts), samples = ncol(counts), share_not_whole_numbers = round(mean(counts != round(counts)), 3))

**Explanation**
- `read_excel(..., sheet = "Detail")` reads the authors' file; `.name_repair = "minimal"` keeps the headers
  as they are.
- `ec` holds the positions and GSM identifiers of the 25 "Expected count" columns (notebook 11).
  `x[, ec$position]` takes those columns by position, so no header is used or printed.
- `dimnames(counts) <- list(x$GeneID, ec$gsm)` names the rows by the file's gene symbols (column `GeneID`)
  and the columns by GSM identifier. `counts[, meta$sample]` puts the samples in the order of the sample
  table.
- `stopifnot()` checks that there are no missing values, no negative values and no repeated gene names.
- The last line gives the numbers of genes and samples, and the share of values that are not whole numbers.
  The file's expected counts are not all whole numbers.

## Name the genes with current NCBI symbols

In [ ]:
gi <- read.delim(file.path(NCBI, "Homo_sapiens.gene_info.gz"), quote = "", colClasses = "character")
syn <- strsplit(gi$Synonyms, "|", fixed = TRUE)
syn_long <- data.frame(synonym = unlist(syn), symbol = rep(gi$Symbol, lengths(syn)))
syn_long <- syn_long[syn_long$synonym != "-" & !duplicated(syn_long), ]
syn_once <- syn_long[!syn_long$synonym %in% syn_long$synonym[duplicated(syn_long$synonym)], ]
old  <- rownames(counts)
sym  <- ifelse(old %in% gi$Symbol, old, syn_once$symbol[match(old, syn_once$synonym)])
c(current_symbol = sum(old %in% gi$Symbol),
  replaced_by_unique_synonym = sum(!old %in% gi$Symbol & !is.na(sym)),
  dropped = sum(is.na(sym)))

**Explanation**
- `gi` is NCBI's gene_info table for human genes (see notebook 03b). Its column `Symbol` is the current
  symbol; its column `Synonyms` lists other names, separated by `|`, or `-` if none.
- `strsplit(..., "|", fixed = TRUE)` splits each gene's synonyms; `syn_long` has one row per synonym and
  its gene's current symbol. For example, a gene with symbol `ABC1` and synonyms `X1|X2` gives two rows,
  `X1 ABC1` and `X2 ABC1`.
- `syn_once` keeps only the synonyms that belong to exactly one gene. A synonym shared by two genes cannot
  say which gene is meant.
- `sym`: a name that is a current symbol is kept; otherwise it is replaced by the gene whose unique synonym it
  is; otherwise it is `NA`.
- The last line counts the three cases.

**Result.** Of the file's 26,506 gene names, 23,743 are current NCBI symbols, 1,557 are replaced by the
current symbol of the one gene that lists them as a synonym, and 1,206 are dropped.

**The replaced names** (the authors' name and the current NCBI symbol).

In [ ]:
rep_tab <- data.frame(authors_name = old, current_symbol = sym)[!old %in% gi$Symbol & !is.na(sym), ]
rownames(rep_tab) <- NULL
nrow(rep_tab); head(rep_tab, 20)
write.csv(rep_tab, art("GSE232381", "symbol_replacements.csv"), row.names = FALSE)

**Explanation**
- `rep_tab` lists every replaced name with its current symbol; the first 20 are shown, and the full list is
  written to `symbol_replacements.csv` in `data/run_artifacts/GSE232381/`.

**Check.** Names that now share a symbol: their counts are summed into one row.

In [ ]:
counts <- counts[!is.na(sym), ]; sym <- sym[!is.na(sym)]
dup <- unique(sym[duplicated(sym)])
data.frame(symbol = sym[sym %in% dup], authors_name = rownames(counts)[sym %in% dup],
           total_counts = round(rowSums(counts[sym %in% dup, , drop = FALSE])))[order(sym[sym %in% dup]), ]
counts <- rowsum(counts, sym)
c(genes = nrow(counts), unique_names = length(unique(rownames(counts))))

**Explanation**
- Names with no current symbol are dropped.
- `dup` holds the symbols that now occur more than once; the table shows each with the authors' names and
  their total counts.
- `rowsum(counts, sym)` adds up the rows that share a symbol, giving one row per symbol.

**Result.** 40 of the 25,300 kept rows share a symbol with another row, for example `SEPT4` and `C17orf47`,
both now `SEPTIN4`. After summing, 25,260 genes remain.

**Total counts per sample (library size).**

## Total counts per sample (library size)

In [ ]:
data.frame(sample = colnames(counts), ln_active = meta$ln_active, total_counts_millions = round(colSums(counts) / 1e6, 1))

**Explanation**
- `colSums(counts)` gives each sample's total count; `/ 1e6` gives millions.

## Remove genes with low counts

In [ ]:
all_counts <- counts
keep <- rowMeans(counts < 10) <= 0.90
c(genes_before = nrow(counts), genes_kept = sum(keep))
counts <- counts[keep, ]

**Explanation**
- `all_counts` keeps the counts of all genes before the filter.
- The WGCNA FAQ rule for RNA sequencing counts: remove genes with fewer than 10 counts in more than 90% of
  samples.
- `rowMeans(counts < 10)` gives, for each gene, the share of samples with fewer than 10 counts; `<= 0.90`
  keeps the genes where that share is at most 90%.

**Result.** 13,578 of 25,260 genes pass the filter.

## Normalize: PFlog1pPF

In [ ]:
pf <- function(X) t(t(X) / colSums(X) * mean(colSums(X)))
E  <- pf(log1p(pf(counts)))
round(quantile(E), 2)

**Explanation**
- `pf(X)`, proportional fitting, scales each sample so that its total equals the mean total over samples.
  For example, a sample with 20 million counts and a mean of 25 million is multiplied by 1.25.
  `t()` is used twice so the division runs sample by sample.
- `log1p(x)` is log(1 + x); it keeps zero counts at zero.
- `pf(log1p(pf(counts)))`: proportional fitting, log1p, then proportional fitting again: PFlog1pPF
  (Booeshaghi AS, Hallgrimsdottir IB, Galvez-Merchan A, Pachter L. Depth normalization for single-cell
  genomics count data. *bioRxiv* 2022, doi:10.1101/2022.05.06.490859).
- `quantile(E)` prints the minimum, quartiles and maximum of the normalized values.

**Check.** After PFlog1pPF every sample has the same total.

In [ ]:
range(colSums(E))
saveRDS(list(E = E, counts = counts, all_counts = all_counts, meta = meta), art("GSE232381", "expression.rds"))

**Explanation**
- `range(colSums(E))` gives the smallest and largest sample total; after the final proportional fitting
  they are equal.
- `expression.rds` keeps the normalized values, the filtered counts, all counts before the filter
  (`all_counts`, used in notebook 16b) and the sample table.